# Generate INSTscenario Templates for Top 17 Uncertainty-Function Pairs

This notebook generates realistic uncertainty scenario templates.

The notebook:
1. Extracts the top-ranked pairs from the report
2. Loads implementation details for each function
3. Loads uncertainty type descriptions
4. Loads plausibility assessments from result files (averaging scores across multiple assessment runs)
5. Generates structured templates for each pair

## Import Required Libraries

In [ ]:
import os
import json
import re
import importlib.util
import inspect
from tqdm.notebook import tqdm

# Import the inst_scenario_template module
try:
    from complexity_integration.complexity_scenario_generation.inst_scenario_template import generate_inst_scenario
    print("Successfully imported generate_inst_scenario function")
except ImportError:
    print("Warning: inst_scenario_template.py not found. Please run the cell below to create it.")

Successfully imported generate_inst_scenario function


## Define Helper Functions

In [ ]:
def load_top_ranked_pairs(report_file="reports/top_functions_report.txt", max_rank=17):
    """
    Extract the top N ranked uncertainty-function pairs from the report file.
    
    Args:
        report_file: Path to the report file
        max_rank: Maximum rank to include (inclusive)
        
    Returns:
        List of tuples (domain, function_name, uncertainty_type, normalized_score)
    """
    if not os.path.exists(report_file):
        print(f"Error: Report file {report_file} not found.")
        return []
    
    with open(report_file, 'r') as f:
        content = f.read()
    
    # Pattern to match uncertainty type sections and ranked function entries
    uncertainty_pattern = r"## ([a-z_]+)\n-+\nRank\s+API Function\s+Score\s+Runs\s+-+\n(.*?)(?=\n\n\n|$)"
    function_pattern = r"(\d+)\s+([A-Za-z]+)\.([a-z_]+)\s+([0-9.]+)"
    
    top_pairs = []
    
    # Find all uncertainty type sections
    for match in re.finditer(uncertainty_pattern, content, re.DOTALL):
        uncertainty_type = match.group(1)
        functions_section = match.group(2)
        
        # Find all function entries within this uncertainty type
        for func_match in re.finditer(function_pattern, functions_section):
            rank = int(func_match.group(1))
            if rank <= max_rank:  # Only include if rank is within our limit
                domain = func_match.group(2)
                function_name = func_match.group(3)
                normalized_score = float(func_match.group(4))
                top_pairs.append((domain, function_name, uncertainty_type, normalized_score))
    
    # Sort by score in descending order
    top_pairs.sort(key=lambda x: x[3], reverse=True)
    return top_pairs


def load_uncertainty_type_details(type_name):
    """
    Load details for a specific uncertainty type from uncertainty_types_reference.py.
    """
    # Load the uncertainty_types_reference module
    spec = importlib.util.spec_from_file_location(
        "uncertainty_types_reference", "complexity_integration/complexity_matching/uncertainty_types_reference.py")
    uncertainty_types = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(uncertainty_types)
    
    # Get the uncertainty type details
    uncertainty_type = getattr(uncertainty_types, type_name.upper(), None)
    if not uncertainty_type:
        print(f"Error: Uncertainty type {type_name} not found.")
        return None
    
    # Format details for the template
    details = {
        "name": uncertainty_type["name"],
        "description": uncertainty_type["description"],
        "criteria": "\n".join([f"{i+1}. {criterion['name']}: {criterion['definition']}" 
                             for i, criterion in enumerate(uncertainty_type["criteria"])])
    }
    
    return details


def load_api_function_details(domain, function_name):
    """
    Load implementation details for a specific API function.
    """
    # Determine the file path for the function implementation
    function_path = f"{domain}/tools/{function_name}.py"
    if not os.path.exists(function_path):
        print(f"Error: Function implementation file {function_path} not found.")
        return None
    
    try:
        # Load the module containing the function
        spec = importlib.util.spec_from_file_location(
            f"{domain}.tools.{function_name}", function_path)
        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)
        
        # Find the class that implements the function
        class_name = ''.join(word.capitalize() for word in function_name.split('_'))
        tool_class = getattr(module, class_name, None)
        if not tool_class:
            print(f"Error: Tool class {class_name} not found in {function_path}.")
            return None
        
        # Get the invoke method and its source code
        invoke_method = getattr(tool_class, "invoke", None)
        if not invoke_method:
            print(f"Error: invoke method not found in {class_name}.")
            return None
        
        # Get the get_info method for description
        get_info_method = getattr(tool_class, "get_info", None)
        if not get_info_method:
            print(f"Error: get_info method not found in {class_name}.")
            return None
        
        # Get the implementation source code and signature
        implementation = inspect.getsource(invoke_method)
        signature = implementation.split("def invoke")[1].split(":")[0] + ":"
        
        # Get the function description from get_info
        info = get_info_method()
        # description = info["function"]["description"]
        description = str(info["function"])
        
        # Format details for the template
        details = {
            "domain": domain,
            "name": function_name,
            "description": description,
            "implementation": implementation,
            "signature": f"def invoke{signature}"
        }
        
        return details
    
    except Exception as e:
        print(f"Error loading function details for {domain}.{function_name}: {str(e)}")
        return None


def load_plausibility_assessment(domain, function_name, uncertainty_type):
    """
    Load plausibility assessment for a specific function-uncertainty pair.
    
    This function computes the average normalized score and concatenates summaries
    across all available assessment result files.
    """
    result_dirs = ["api_assessment_results_0", "api_assessment_results_1", "api_assessment_results_2"]
    found_results = []
    summaries = []
    scores = []
    
    # Find all available assessment results
    for result_dir in result_dirs:
        result_path = f"{result_dir}/{domain}/{function_name}/{uncertainty_type}_result.json"
        if os.path.exists(result_path):
            try:
                with open(result_path, 'r') as f:
                    result = json.load(f)
                    found_results.append(result)
                    
                    # Extract summary from prediction
                    summary_match = re.search(r"## Overall Assessment.*?- Summary:(.*?)(?=\n\n|$)", 
                                             result["prediction"], re.DOTALL)
                    if summary_match:
                        summaries.append(f"[From {result_dir}]: {summary_match.group(1).strip()}")
                    
                    # Get normalized score
                    if "normalized_score" in result:
                        scores.append(float(result["normalized_score"]))
            except Exception as e:
                print(f"Error parsing assessment file {result_path}: {str(e)}")
    
    if not found_results:
        print(f"Warning: No assessment results found for {domain}.{function_name} with {uncertainty_type}.")
        # Return placeholder assessment
        return {
            "summary": "Assessment summary not found.",
            "normalized_score": "N/A",
            "likelihood": "Unknown"
        }
    
    # Calculate average score and determine likelihood
    avg_score = sum(scores) / len(scores) if scores else 0
    
    # Determine likelihood based on average score
    if avg_score >= 0.67:
        likelihood = "High"
    elif avg_score >= 0.33:
        likelihood = "Moderate"
    else:
        likelihood = "Low"
    
    # Format assessment details
    assessment = {
        "summary": "\n\n".join(summaries),
        "normalized_score": f"{avg_score:.3f}",
        "likelihood": likelihood
    }
    
    return assessment


def generate_scenario_for_pair(domain, function_name, uncertainty_type, output_dir="inst_scenarios"):
    """
    Generate INSTscenario template for a specific function-uncertainty pair.
    """
    print(f"Generating scenario for {domain}.{function_name} with {uncertainty_type}...")
    
    # Load all required details
    api_function_details = load_api_function_details(domain, function_name)
    uncertainty_type_details = load_uncertainty_type_details(uncertainty_type)
    plausibility_assessment = load_plausibility_assessment(domain, function_name, uncertainty_type)
    
    if not api_function_details or not uncertainty_type_details:
        print("Failed to generate scenario due to missing function or uncertainty type information.")
        return None
    
    try:
        # Generate the scenario template
        scenario = generate_inst_scenario(
            api_function_details, 
            uncertainty_type_details, 
            plausibility_assessment
        )
        
        # Save to output file
        os.makedirs(output_dir, exist_ok=True)
        output_path = f"{output_dir}/{domain}_{function_name}__{uncertainty_type}.md"
        
        with open(output_path, 'w') as f:
            f.write(scenario)
        
        print(f"Scenario template saved to {output_path}")
        return output_path
    
    except Exception as e:
        print(f"Error generating scenario for {domain}.{function_name} with {uncertainty_type}: {str(e)}")
        return None

## Load Top Ranked Function-Uncertainty Pairs

In [3]:
# Configure output directory
output_dir = "inst_scenarios"
os.makedirs(output_dir, exist_ok=True)

# Load the top ranked pairs
top_pairs = load_top_ranked_pairs(max_rank=34)
print(f"Loaded {len(top_pairs)} function-uncertainty pairs with rank <= 34.")

# Show the top 10 pairs to verify
print("\nTop 10 uncertainty-function pairs by score:")
for i, (domain, function_name, uncertainty_type, score) in enumerate(top_pairs[:10]):
    print(f"{i+1}. {domain}.{function_name} with {uncertainty_type} (score: {score})")

Loaded 306 function-uncertainty pairs with rank <= 34.

Top 10 uncertainty-function pairs by score:
1. CulinaryControlEnv.get_meal_suggestions with ambiguous_documentation (score: 1.0)
2. CulinaryControlEnv.create_custom_recipe with ambiguous_documentation (score: 1.0)
3. CulinaryControlEnv.create_meal_plan with ambiguous_documentation (score: 1.0)
4. MediaControlEnv.resume with complex_dependency_chains (score: 1.0)
5. CulinaryControlEnv.place_delivery_order with complex_dependency_chains (score: 1.0)
6. TransactionEnv.checkout with complex_dependency_chains (score: 1.0)
7. MediaControlEnv.play with complex_dependency_chains (score: 0.958)
8. CulinaryControlEnv.place_delivery_order with unclear_functionality_boundaries (score: 0.943)
9. MediaControlEnv.next with complex_dependency_chains (score: 0.917)
10. SmartHomeEnv.volume_adjust with ad_hoc_rules (score: 0.9)


In [4]:
top_pairs

[('CulinaryControlEnv',
  'get_meal_suggestions',
  'ambiguous_documentation',
  1.0),
 ('CulinaryControlEnv',
  'create_custom_recipe',
  'ambiguous_documentation',
  1.0),
 ('CulinaryControlEnv', 'create_meal_plan', 'ambiguous_documentation', 1.0),
 ('MediaControlEnv', 'resume', 'complex_dependency_chains', 1.0),
 ('CulinaryControlEnv',
  'place_delivery_order',
  'complex_dependency_chains',
  1.0),
 ('TransactionEnv', 'checkout', 'complex_dependency_chains', 1.0),
 ('MediaControlEnv', 'play', 'complex_dependency_chains', 0.958),
 ('CulinaryControlEnv',
  'place_delivery_order',
  'unclear_functionality_boundaries',
  0.943),
 ('MediaControlEnv', 'next', 'complex_dependency_chains', 0.917),
 ('SmartHomeEnv', 'volume_adjust', 'ad_hoc_rules', 0.9),
 ('SmartHomeEnv', 'color_set', 'ambiguous_documentation', 0.9),
 ('CulinaryControlEnv', 'search_recipes', 'ambiguous_documentation', 0.9),
 ('CulinaryControlEnv', 'search_restaurants', 'ambiguous_documentation', 0.9),
 ('MediaControlEnv',
 

## Select Function-Uncertainty Pairs to Generate

You can choose to generate scenarios for all pairs or a specific subset.

In [5]:
# Define pairs to generate
# Set to None to generate all pairs
pairs_to_generate = None

# If you want to generate only specific pairs, use:
# pairs_to_generate = [
#     ("SmartHomeEnv", "volume_adjust", "ad_hoc_rules"),
#     ("MediaControlEnv", "play", "ad_hoc_rules"),
#     ("TimeNotificationEnv", "create_alarm", "ad_hoc_rules")
# ]

# Get the list of pairs to process
pairs_to_process = []
if pairs_to_generate is None:
    pairs_to_process = top_pairs
else:
    # Find the specified pairs in the top_pairs list to get their scores
    for domain, function_name, uncertainty_type in pairs_to_generate:
        for d, f, u, score in top_pairs:
            if d == domain and f == function_name and u == uncertainty_type:
                pairs_to_process.append((d, f, u, score))
                break

print(f"Will generate scenarios for {len(pairs_to_process)} function-uncertainty pairs.")

Will generate scenarios for 306 function-uncertainty pairs.


## Generate Scenarios for Selected Pairs

This cell generates the scenario templates for each selected pair. It may take some time to complete.

In [6]:
pairs_to_process

[('CulinaryControlEnv',
  'get_meal_suggestions',
  'ambiguous_documentation',
  1.0),
 ('CulinaryControlEnv',
  'create_custom_recipe',
  'ambiguous_documentation',
  1.0),
 ('CulinaryControlEnv', 'create_meal_plan', 'ambiguous_documentation', 1.0),
 ('MediaControlEnv', 'resume', 'complex_dependency_chains', 1.0),
 ('CulinaryControlEnv',
  'place_delivery_order',
  'complex_dependency_chains',
  1.0),
 ('TransactionEnv', 'checkout', 'complex_dependency_chains', 1.0),
 ('MediaControlEnv', 'play', 'complex_dependency_chains', 0.958),
 ('CulinaryControlEnv',
  'place_delivery_order',
  'unclear_functionality_boundaries',
  0.943),
 ('MediaControlEnv', 'next', 'complex_dependency_chains', 0.917),
 ('SmartHomeEnv', 'volume_adjust', 'ad_hoc_rules', 0.9),
 ('SmartHomeEnv', 'color_set', 'ambiguous_documentation', 0.9),
 ('CulinaryControlEnv', 'search_recipes', 'ambiguous_documentation', 0.9),
 ('CulinaryControlEnv', 'search_restaurants', 'ambiguous_documentation', 0.9),
 ('MediaControlEnv',
 

In [7]:
# Generate scenarios for each pair
generated_files = []

for domain, function_name, uncertainty_type, score in tqdm(pairs_to_process, desc="Generating scenarios"):
    output_path = generate_scenario_for_pair(domain, function_name, uncertainty_type, output_dir)
    if output_path:
        generated_files.append((output_path, score))



Generating scenarios:   0%|          | 0/306 [00:00<?, ?it/s]

Generating scenario for CulinaryControlEnv.get_meal_suggestions with ambiguous_documentation...
Scenario template saved to inst_scenarios/CulinaryControlEnv_get_meal_suggestions__ambiguous_documentation.md
Generating scenario for CulinaryControlEnv.create_custom_recipe with ambiguous_documentation...
Scenario template saved to inst_scenarios/CulinaryControlEnv_create_custom_recipe__ambiguous_documentation.md
Generating scenario for CulinaryControlEnv.create_meal_plan with ambiguous_documentation...
Scenario template saved to inst_scenarios/CulinaryControlEnv_create_meal_plan__ambiguous_documentation.md
Generating scenario for MediaControlEnv.resume with complex_dependency_chains...
Scenario template saved to inst_scenarios/MediaControlEnv_resume__complex_dependency_chains.md
Generating scenario for CulinaryControlEnv.place_delivery_order with complex_dependency_chains...
Scenario template saved to inst_scenarios/CulinaryControlEnv_place_delivery_order__complex_dependency_chains.md
Gene

## View Results

Display the generated scenario templates, sorted by score.

In [8]:
print(f"\nGenerated {len(generated_files)} scenario templates.")
print("\nTop 20 generated scenario templates by score:")

for file_path, score in sorted(generated_files, key=lambda x: x[1], reverse=True)[:20]:
    print(f"- {file_path} (score: {score})")

print(f"\nTotal scenarios generated: {len(generated_files)}")


Generated 306 scenario templates.

Top 20 generated scenario templates by score:
- inst_scenarios/CulinaryControlEnv_get_meal_suggestions__ambiguous_documentation.md (score: 1.0)
- inst_scenarios/CulinaryControlEnv_create_custom_recipe__ambiguous_documentation.md (score: 1.0)
- inst_scenarios/CulinaryControlEnv_create_meal_plan__ambiguous_documentation.md (score: 1.0)
- inst_scenarios/MediaControlEnv_resume__complex_dependency_chains.md (score: 1.0)
- inst_scenarios/CulinaryControlEnv_place_delivery_order__complex_dependency_chains.md (score: 1.0)
- inst_scenarios/TransactionEnv_checkout__complex_dependency_chains.md (score: 1.0)
- inst_scenarios/MediaControlEnv_play__complex_dependency_chains.md (score: 0.958)
- inst_scenarios/CulinaryControlEnv_place_delivery_order__unclear_functionality_boundaries.md (score: 0.943)
- inst_scenarios/MediaControlEnv_next__complex_dependency_chains.md (score: 0.917)
- inst_scenarios/SmartHomeEnv_volume_adjust__ad_hoc_rules.md (score: 0.9)
- inst_scena

## Explore a Generated Scenario (Optional)

View the contents of a specific generated scenario template.

In [9]:
# To view a specific scenario, uncomment and run this cell

# scenario_path = "inst_scenarios/SmartHomeEnv_volume_adjust__ad_hoc_rules.md"  # Change this path as needed
scenario_path = "inst_scenarios/CommunicationController_get_call_history__partially_irrelevant_information.md"
if os.path.exists(scenario_path):
    with open(scenario_path, 'r') as f:
        scenario_content = f.read()
    print(scenario_content)
else:
    print(f"Scenario file {scenario_path} not found.")

# Realistic Uncertainty Scenario: Partially Irrelevant Information in CommunicationController.get_call_history

## Task

Specify a concrete, realistic scenario where the uncertainty type 'Partially Irrelevant Information' 
would manifest in the API function 'CommunicationController.get_call_history' 
in production environments. Focus on converting the abstract uncertainty type into specific, 
practical manifestations that API users might encounter.

For each manifestation, modify the API Description and Implementation to realistically demonstrate
this uncertainty, making only the minimum necessary changes and clearly marking your modifications.

## API Function Information

### Description
{'name': 'get_call_history', 'description': "Get call history for the current user. This tool retrieves the user's call records, including incoming and outgoing calls, with details such as duration and status.", 'parameters': {'type': 'object', 'properties': {'time_range': {'type': 'string', 'descrip

In [10]:
# To view a specific scenario, uncomment and run this cell

# scenario_path = "inst_scenarios/SmartHomeEnv_volume_adjust__ad_hoc_rules.md"  # Change this path as needed
scenario_path = "inst_scenarios/CulinaryControlEnv_get_restaurant_menu__system_failure_error.md"
if os.path.exists(scenario_path):
    with open(scenario_path, 'r') as f:
        scenario_content = f.read()
    print(scenario_content)
else:
    print(f"Scenario file {scenario_path} not found.")

# Realistic Uncertainty Scenario: System Failure Error in CulinaryControlEnv.get_restaurant_menu

## Task

Specify a concrete, realistic scenario where the uncertainty type 'System Failure Error' 
would manifest in the API function 'CulinaryControlEnv.get_restaurant_menu' 
in production environments. Focus on converting the abstract uncertainty type into specific, 
practical manifestations that API users might encounter.

For each manifestation, modify the API Description and Implementation to realistically demonstrate
this uncertainty, making only the minimum necessary changes and clearly marking your modifications.

## API Function Information

### Description
{'name': 'get_restaurant_menu', 'description': 'Get the complete menu for a specific restaurant, including item details, prices, and categories.', 'parameters': {'type': 'object', 'properties': {'restaurant_id': {'type': 'string', 'description': 'The unique identifier of the restaurant to retrieve menu for.'}}, 'required': ['re

In [11]:
# To view a specific scenario, uncomment and run this cell

# scenario_path = "inst_scenarios/SmartHomeEnv_volume_adjust__ad_hoc_rules.md"  # Change this path as needed
scenario_path = "inst_scenarios/CulinaryControlEnv_place_delivery_order__informational_notice.md"
if os.path.exists(scenario_path):
    with open(scenario_path, 'r') as f:
        scenario_content = f.read()
    print(scenario_content)
else:
    print(f"Scenario file {scenario_path} not found.")

# Realistic Uncertainty Scenario: Informational Notice in CulinaryControlEnv.place_delivery_order

## Task

Specify a concrete, realistic scenario where the uncertainty type 'Informational Notice' 
would manifest in the API function 'CulinaryControlEnv.place_delivery_order' 
in production environments. Focus on converting the abstract uncertainty type into specific, 
practical manifestations that API users might encounter.

For each manifestation, modify the API Description and Implementation to realistically demonstrate
this uncertainty, making only the minimum necessary changes and clearly marking your modifications.

## API Function Information

### Description
{'name': 'place_delivery_order', 'description': 'Place a food delivery order from a restaurant. The order will be processed and delivered to the specified address.', 'parameters': {'type': 'object', 'properties': {'restaurant_id': {'type': 'string', 'description': 'The unique identifier of the restaurant to order from.'}, 'it

In [12]:
# To view a specific scenario, uncomment and run this cell

# scenario_path = "inst_scenarios/SmartHomeEnv_volume_adjust__ad_hoc_rules.md"  # Change this path as needed
scenario_path = "inst_scenarios/CulinaryControlEnv_place_delivery_order__feature_limitation_error.md"
if os.path.exists(scenario_path):
    with open(scenario_path, 'r') as f:
        scenario_content = f.read()
    print(scenario_content)
else:
    print(f"Scenario file {scenario_path} not found.")

# Realistic Uncertainty Scenario: Feature Limitation Error in CulinaryControlEnv.place_delivery_order

## Task

Specify a concrete, realistic scenario where the uncertainty type 'Feature Limitation Error' 
would manifest in the API function 'CulinaryControlEnv.place_delivery_order' 
in production environments. Focus on converting the abstract uncertainty type into specific, 
practical manifestations that API users might encounter.

For each manifestation, modify the API Description and Implementation to realistically demonstrate
this uncertainty, making only the minimum necessary changes and clearly marking your modifications.

## API Function Information

### Description
{'name': 'place_delivery_order', 'description': 'Place a food delivery order from a restaurant. The order will be processed and delivered to the specified address.', 'parameters': {'type': 'object', 'properties': {'restaurant_id': {'type': 'string', 'description': 'The unique identifier of the restaurant to order from

## View Assessment Summaries Comparison (Optional)

Compare the summaries from different assessment runs for a specific function-uncertainty pair.

In [13]:
def show_assessment_summaries(domain, function_name, uncertainty_type):
    """Show the assessment summaries for a specific function-uncertainty pair."""
    result_dirs = ["api_assessment_results_0", "api_assessment_results_1", "api_assessment_results_2"]
    
    print(f"Assessment summaries for {domain}.{function_name} with {uncertainty_type}:\n")
    
    found_any = False
    for result_dir in result_dirs:
        result_path = f"{result_dir}/{domain}/{function_name}/{uncertainty_type}_result.json"
        if os.path.exists(result_path):
            found_any = True
            try:
                with open(result_path, 'r') as f:
                    result = json.load(f)
                
                print(f"### From {result_dir}:")
                print(f"Score: {result.get('normalized_score', 'N/A')} ({result.get('likelihood', 'Unknown')})")
                
                summary_match = re.search(r"## Overall Assessment.*?- Summary:(.*?)(?=\n\n|$)", 
                                         result.get("prediction", ""), re.DOTALL)
                if summary_match:
                    print(f"Summary: {summary_match.group(1).strip()}\n")
                else:
                    print("Summary not found.\n")
                    
            except Exception as e:
                print(f"Error parsing {result_path}: {str(e)}\n")
    
    if not found_any:
        print("No assessment results found.")

# Example usage (uncomment to run):
show_assessment_summaries("SmartHomeEnv", "volume_adjust", "ad_hoc_rules")

Assessment summaries for SmartHomeEnv.volume_adjust with ad_hoc_rules:

### From api_assessment_results_0:
Score: 0.9 (High (0.67-1.0))
Summary: Volume adjustment functions inherently develop ad hoc rules due to the heterogeneous nature of audio devices and their control protocols. The function must accommodate a wide variety of device-specific behaviors, non-linear volume scales, and special value interpretations across different manufacturers and device generations. These complexities naturally lead to numerous special cases and hidden constraints that aren't immediately obvious from the function's simple description.

### From api_assessment_results_1:
Score: 0.9 (High (0.67-1.0))
Summary: Volume adjustment functions inherently develop ad hoc rules due to the wide variety of audio devices with different behaviors, scales, and limitations. The function must accommodate device-specific quirks, non-linear volume curves, and special value handling that has evolved over decades of audio 